#Step 1: Load data

In [ ]:
#required libraries and paths
import os
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Colab Notebooks/Imperial ML/Capstone/initial_data"

Mounted at /content/drive


In [ ]:
# LOAD Existing data and append as needed

function_data = {}

for n in range(1, 9):
    # read in updated data if it exists
    func_path = f"{base_path}/function_{n}"
    x_updated_path = f"{func_path}/updated_inputs.npy"
    y_updated_path = f"{func_path}/updated_outputs.npy"

    if os.path.exists(x_updated_path) and os.path.exists(y_updated_path):
        inputs = np.load(x_updated_path)
        outputs = np.load(y_updated_path)
    else:
        inputs = np.load(f"{func_path}/initial_inputs.npy")
        outputs = np.load(f"{func_path}/initial_outputs.npy")

    # Add this week's new point if needed
    #x_new = X_new[n - 1].reshape(1, -1)
    #y_new_val = np.array([y_new[n - 1]])

    #X_updated = np.vstack([inputs, x_new])
    #y_updated = np.concatenate([outputs, y_new_val])

    # Save clean updated version
    #np.save(f"{func_path}/updated_inputs.npy", X_updated)
    #np.save(f"{func_path}/updated_outputs.npy", y_updated)

    X_updated = np.vstack([inputs]) #X_updated = np.vstack([inputs, X_updated])
    y_updated = np.concatenate([outputs]) #y_updated = np.concatenate([outputs, y_updated])

    # DataFrame version
    columns = [f"x{i+1}" for i in range(X_updated.shape[1])]
    df = pd.DataFrame(X_updated, columns=columns)
    df["output"] = y_updated

    function_data[f"function_{n}"] = df

print(function_data["function_1"])

          x1        x2         output
0   0.319404  0.762959   1.322677e-79
1   0.574329  0.879898   1.033078e-46
2   0.731024  0.733000   7.710875e-16
3   0.840353  0.264732  3.341771e-124
4   0.650114  0.681526  -3.606063e-03
5   0.410437  0.147554  -2.159249e-54
6   0.312691  0.078723  -2.089093e-91
7   0.683418  0.861057   2.535001e-40
8   0.082507  0.403488   3.606771e-81
9   0.883890  0.582254   6.229856e-48
10  0.705514  0.216141  -5.497122e-87
11  0.372082  0.363786   2.607711e-05
12  0.346346  0.346953   1.630129e-08
13  0.776225  0.154079  1.162057e-139
14  0.388785  0.512744  -6.684744e-08
15  0.389317  0.368762  -3.238835e-04


#Step 2: Define optimiser and configurations

In [ ]:
# import required libraries and packages
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from scipy.stats import norm, qmc

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

# XGBoost is optional.
try:
    from xgboost import XGBRegressor

    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    warnings.warn(
        "xgboost is not installed. The pipeline will run without it."
    )


### Note: Need to specify aquisition function and parameters here - you can leave default kappa = 2.0 and re-consider based on uncertainty diagnostics in step 3

In [ ]:

# 1. CONFIGURATION
#--------------------------------------------------

@dataclass
class FunctionConfig:
    transform: str = "none"

    # Final GP acquisition
    acquisition: str = "ei"       # options are "ei" or "ucb"
    kappa: float = 0.5             # used by UCB
    xi: float = 0.0                # used by EI

    # Candidate-pool composition
    n_global: int = 8192
    n_local: int = 8192
    local_scale: Any = 0.05

    # Neural-network candidate generator
    use_nn_generator: bool = False
    nn_seeds: Sequence[int] = field(
        default_factory=lambda: (11, 22, 33, 44, 55)
    )
    nn_hidden_sizes: Tuple[int, ...] = (16, 16)
    nn_epochs: int = 2000
    nn_learning_rate: float = 0.01
    nn_weight_decay: float = 0.01
    nn_best_starts: int = 5
    nn_sobol_starts: int = 20
    nn_gradient_steps: int = 200
    nn_gradient_learning_rate: float = 0.01

    # GP settings
    gp_noise_level: float = 1e-3
    gp_restarts: int = 10


    # Explicit candidates, such as [1,1,1,1] for exploring the corner solution
    special_candidates: Optional[np.ndarray] = None


FUNCTION_CONFIGS = {
    "function_1": FunctionConfig(
        transform="asinh",
        acquisition="ucb",
        kappa=1.0,
        xi=0.0,
        n_global=8192,
        n_local=8192,
        local_scale=0.04,
        use_nn_generator=False,
    ),

    "function_2": FunctionConfig(
        transform="none",
        acquisition="ucb",
        kappa=2.0,
        xi=0.0,
        n_global=8192,
        n_local=12288,
        local_scale=np.array([0.025, 0.015]),
        use_nn_generator=False,
    ),

    "function_3": FunctionConfig(
        transform="none",
        acquisition="ucb",
        kappa=1.0,
        xi=0.0,
        n_global=8192,
        n_local=8192,
        local_scale=0.10,
        use_nn_generator=True,
    ),

    "function_4": FunctionConfig(
        transform="none",
        acquisition="ucb",
        kappa=1.0,
        xi=0.0,
        n_global=16384,
        n_local=12288,
        local_scale=0.05,
        use_nn_generator=True,
    ),

    "function_5": FunctionConfig(
        transform="log1p",
        acquisition="ucb",
        kappa=1.0,
        xi=0.0,
        n_global=16384,
        n_local=12288,
        local_scale=0.04,
        use_nn_generator=True,
        special_candidates=np.ones((1, 4)),
    ),

    "function_6": FunctionConfig(
        transform="none",
        acquisition="ucb",
        kappa=1.0,
        xi=0.0,
        n_global=16384,
        n_local=12288,
        local_scale=np.array([
            0.05,
            0.05,
            0.05,
            0.05,
            0.005,
        ]),
        use_nn_generator=False,
    ),

    "function_7": FunctionConfig(
        transform="none",
        acquisition="ucb",
        kappa=1.0,
        xi=0.01,
        n_global=32768,
        n_local=12288,
        local_scale=0.06,
        use_nn_generator=False,
    ),

    "function_8": FunctionConfig(
        transform="none",
        acquisition="ucb",
        kappa=1.0,
        xi=0.0,
        n_global=32768,
        n_local=12288,
        local_scale=0.05,
        use_nn_generator=True,
    ),

}

# Step 3: Define the main engine for the optimiser,   generate candidates, and alternative models to compare

In [ ]:
# 2. OUTPUT TRANSFORMATIONS
#--------------------------------------------------

def fit_output_transform(
    y: np.ndarray,
    transform: str,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Transform outputs before surrogate fitting.
    """
    y = np.asarray(y, dtype=float)

    if transform == "none":
        return y.copy(), {
            "name": "none",
            "scale": None,
        }

    if transform == "log1p":
        if np.any(y <= -1):
            raise ValueError(
                "log1p requires all y values to be greater than -1."
            )

        return np.log1p(y), {
            "name": "log1p",
            "scale": None,
        }

    if transform == "asinh":
        centre = np.median(y)
        scale = np.median(
            np.abs(y - centre)
        )

        scale = max(
            float(scale),
            1e-6,
        )

        return np.arcsinh((y - centre) / scale), {
            "name": "asinh",
            "centre": float(centre),
            "scale": scale,
        }

    raise ValueError(
        "transform must be 'none', 'log1p', or 'asinh'."
    )


def inverse_output_transform(
    y_transformed: np.ndarray,
    transform_info: Dict[str, Any],
) -> np.ndarray:
    """
    Inverse-transform predictions for interpretation.
    """
    y_transformed = np.asarray(
        y_transformed,
        dtype=float,
    )

    name = transform_info["name"]

    if name == "none":
        return y_transformed

    if name == "log1p":
        return np.expm1(y_transformed)

    if name == "asinh":
        return (
            transform_info["centre"]
            + transform_info["scale"]
            * np.sinh(y_transformed)
        )

    raise ValueError(
        f"Unknown transformation: {name}"
    )


def extract_xy(
    df: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    Extract input matrix and output vector.
    """
    if "output" not in df.columns:
        raise ValueError(
            "DataFrame must contain an 'output' column."
        )

    input_columns = [
        col for col in df.columns
        if col != "output"
    ]

    X = df[
        input_columns
    ].to_numpy(dtype=float)

    y = df[
        "output"
    ].to_numpy(dtype=float)

    if X.ndim != 2:
        raise ValueError(
            "Inputs must form a two-dimensional matrix."
        )

    if len(X) != len(y):
        raise ValueError(
            "X and y have inconsistent row counts."
        )

    if np.any(~np.isfinite(X)):
        raise ValueError(
            "Input data contain NaN or infinite values."
        )

    if np.any(~np.isfinite(y)):
        raise ValueError(
            "Output data contain NaN or infinite values."
        )

    if np.any((X < 0) | (X > 1)):
        raise ValueError(
            "All input values must lie in [0, 1]."
        )

    return X, y, input_columns


# 3. GAUSSIAN PROCESS
#--------------------------------------------------

def fit_gp(
    X: np.ndarray,
    y_model: np.ndarray,
    noise_level: float,
    n_restarts: int,
    random_state: int,
) -> GaussianProcessRegressor:
    """
    Fit a Matérn-3/2 ARD Gaussian Process.
    """
    dim = X.shape[1]

    kernel = (
        ConstantKernel(
            1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.ones(dim),
            length_scale_bounds=[
                (1e-2, 10.0)
            ] * dim,
            nu=1.5,
        )
        + WhiteKernel(
            noise_level=noise_level,
            noise_level_bounds=(1e-6, 1e-1),
        )
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=n_restarts,
        random_state=random_state,
    )

    # Fit directly to the maximisation target.
    gp.fit(X, y_model)

    return gp


def extract_gp_lengthscales(
    gp: GaussianProcessRegressor,
) -> np.ndarray:
    """
    Extract ARD length scales from the fitted kernel.
    """
    try:
        return np.atleast_1d(
            gp.kernel_.k1.k2.length_scale
        ).astype(float)

    except (AttributeError, IndexError):
        return np.array([])



# 5. ACQUISITION FUNCTIONS
#--------------------------------------------------

def expected_improvement(
    mu: np.ndarray,
    sigma: np.ndarray,
    incumbent: float,
    xi: float = 0.0,
) -> np.ndarray:
    """
    Expected Improvement for maximisation.
    """
    mu = np.asarray(mu, dtype=float)
    sigma = np.asarray(sigma, dtype=float)

    improvement = (
        mu - incumbent - xi
    )

    safe_sigma = np.maximum(
        sigma,
        1e-12,
    )

    z = improvement / safe_sigma

    ei = (
        improvement * norm.cdf(z)
        + safe_sigma * norm.pdf(z)
    )

    ei = np.where(
        sigma > 1e-12,
        ei,
        0.0,
    )

    return np.maximum(ei, 0.0)


def upper_confidence_bound(
    mu: np.ndarray,
    sigma: np.ndarray,
    kappa: float,
) -> np.ndarray:
    """
    UCB for maximisation.
    """
    return (
        np.asarray(mu)
        + kappa * np.asarray(sigma)
    )


# 5. GENERATE CANDIDATEs
#--------------------------------------------------

def reflect_to_unit_interval(
    x: np.ndarray,
) -> np.ndarray:
    """
    Reflect out-of-bounds values back into [0,1].

    Avoids the boundary pile-up caused by np.clip.
    """
    x = np.mod(x, 2.0)

    return np.where(
        x <= 1.0,
        x,
        2.0 - x,
    )


def generate_sobol_candidates(
    dim: int,
    n_candidates: int,
    random_state: int,
) -> np.ndarray:
    """
    Generate global Sobol candidates.
    """
    if n_candidates <= 0:
        return np.empty((0, dim))

    sampler = qmc.Sobol(
        d=dim,
        scramble=True,
        seed=random_state,
    )

    return sampler.random(
        n=n_candidates
    )


def generate_local_candidates(
    x_best: np.ndarray,
    n_candidates: int,
    scale: Any,
    random_state: int,
) -> np.ndarray:
    """
    Generate local Gaussian perturbations around incumbent.
    """
    x_best = np.asarray(
        x_best,
        dtype=float,
    )

    dim = len(x_best)

    if n_candidates <= 0:
        return np.empty((0, dim))

    scale = np.asarray(
        scale,
        dtype=float,
    )

    if scale.ndim == 0:
        scale = np.repeat(
            scale,
            dim,
        )

    if len(scale) != dim:
        raise ValueError(
            "local_scale must be scalar or match input dimension."
        )

    rng = np.random.default_rng(
        random_state
    )

    candidates = (
        x_best
        + rng.normal(
            loc=0.0,
            scale=scale,
            size=(n_candidates, dim),
        )
    )

    return reflect_to_unit_interval(
        candidates
    )


def combine_candidate_sources(
    candidate_groups: Sequence[
        Tuple[str, np.ndarray]
    ],
    decimals: int = 10,
) -> pd.DataFrame:
    """
    Combine and deduplicate candidates while retaining provenance.
    """
    frames = []

    for source, points in candidate_groups:
        points = np.asarray(
            points,
            dtype=float,
        )

        if len(points) == 0:
            continue

        dim = points.shape[1]

        frame = pd.DataFrame(
            points,
            columns=[
                f"x{i + 1}"
                for i in range(dim)
            ],
        )

        frame["source"] = source
        frames.append(frame)

    if not frames:
        raise ValueError(
            "No candidate points were generated."
        )

    candidates = pd.concat(
        frames,
        ignore_index=True,
    )

    input_columns = [
        col for col in candidates.columns
        if col.startswith("x")
    ]

    candidates[input_columns] = (
        candidates[input_columns]
        .clip(0.0, 1.0)
        .round(decimals)
    )

    candidates = (
        candidates
        .groupby(
            input_columns,
            as_index=False,
        )
        .agg({
            "source": lambda values: "|".join(
                sorted(set(values))
            )
        })
    )

    return candidates



# 7. SMALL PYTORCH NEURAL NETWORK
#--------------------------------------------------

class SmallNN(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_sizes: Tuple[int, ...],
    ):
        super().__init__()

        layers = []
        previous_size = input_dim

        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(
                    previous_size,
                    hidden_size,
                ),
                nn.Tanh(),
            ])

            previous_size = hidden_size

        layers.append(
            nn.Linear(
                previous_size,
                1,
            )
        )

        self.network = nn.Sequential(
            *layers
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.network(x)


@dataclass
class FittedNN:
    model: SmallNN
    y_mean: float
    y_std: float
    seed: int


def fit_single_nn(
    X: np.ndarray,
    y_model: np.ndarray,
    seed: int,
    hidden_sizes: Tuple[int, ...],
    epochs: int,
    learning_rate: float,
    weight_decay: float,
) -> FittedNN:
    """
    Fit one small full-batch neural network.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    y_model = np.asarray(
        y_model,
        dtype=np.float32,
    )

    y_mean = float(
        np.mean(y_model)
    )

    y_std = max(
        float(np.std(y_model)),
        1e-8,
    )

    y_standardised = (
        y_model - y_mean
    ) / y_std

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
    )

    y_tensor = torch.tensor(
        y_standardised[:, None],
        dtype=torch.float32,
    )

    model = SmallNN(
        input_dim=X.shape[1],
        hidden_sizes=hidden_sizes,
    )

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    criterion = nn.MSELoss()

    model.train()

    for _ in range(epochs):
        optimiser.zero_grad()

        prediction = model(
            X_tensor
        )

        loss = criterion(
            prediction,
            y_tensor,
        )

        loss.backward()
        optimiser.step()

    return FittedNN(
        model=model,
        y_mean=y_mean,
        y_std=y_std,
        seed=seed,
    )


def fit_nn_ensemble(
    X: np.ndarray,
    y_model: np.ndarray,
    seeds: Sequence[int],
    hidden_sizes: Tuple[int, ...],
    epochs: int,
    learning_rate: float,
    weight_decay: float,
) -> List[FittedNN]:
    """
    Fit multiple NNs across random seeds.
    """
    return [
        fit_single_nn(
            X=X,
            y_model=y_model,
            seed=seed,
            hidden_sizes=hidden_sizes,
            epochs=epochs,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )
        for seed in seeds
    ]


def predict_single_nn(
    fitted_nn: FittedNN,
    X: np.ndarray,
) -> np.ndarray:
    """
    Predict in transformed-output units.
    """
    fitted_nn.model.eval()

    X_tensor = torch.tensor(
        np.asarray(
            X,
            dtype=np.float32,
        ),
        dtype=torch.float32,
    )

    with torch.no_grad():
        prediction_standardised = (
            fitted_nn.model(
                X_tensor
            )
            .squeeze(-1)
            .cpu()
            .numpy()
        )

    return (
        prediction_standardised
        * fitted_nn.y_std
        + fitted_nn.y_mean
    )


def predict_nn_ensemble(
    ensemble: Sequence[FittedNN],
    X: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return mean and seed-to-seed standard deviation.
    """
    predictions = np.column_stack([
        predict_single_nn(
            fitted_nn,
            X,
        )
        for fitted_nn in ensemble
    ])

    return (
        np.mean(
            predictions,
            axis=1,
        ),
        np.std(
            predictions,
            axis=1,
        ),
    )



# 7. NN GRADIENT-BASED CANDIDATE GENERATION
#--------------------------------------------------

def optimise_nn_input(
    fitted_nn: FittedNN,
    x_start: np.ndarray,
    n_steps: int,
    learning_rate: float,
) -> np.ndarray:
    """
    Maximise one NN surrogate with respect to its input.
    """
    model = fitted_nn.model
    model.eval()

    x = torch.tensor(
        np.asarray(
            x_start,
            dtype=np.float32,
        ),
        dtype=torch.float32,
        requires_grad=True,
    )

    optimiser = torch.optim.Adam(
        [x],
        lr=learning_rate,
    )

    for _ in range(n_steps):
        optimiser.zero_grad()

        prediction_standardised = model(
            x.unsqueeze(0)
        ).squeeze()

        loss = -prediction_standardised

        loss.backward()
        optimiser.step()

        with torch.no_grad():
            x.clamp_(0.0, 1.0)

    return (
        x.detach()
        .cpu()
        .numpy()
        .astype(float)
    )


def generate_nn_candidates(
    ensemble: Sequence[FittedNN],
    X: np.ndarray,
    y_original: np.ndarray,
    n_best_starts: int,
    n_sobol_starts: int,
    n_steps: int,
    learning_rate: float,
    random_state: int,
) -> np.ndarray:
    """
    Optimise each NN from multiple observed and Sobol starts.
    """
    dim = X.shape[1]

    n_best_starts = min(
        n_best_starts,
        len(X),
    )

    best_indices = np.argsort(
        y_original
    )[-n_best_starts:]

    observed_starts = X[
        best_indices
    ]

    sobol_starts = (
        generate_sobol_candidates(
            dim=dim,
            n_candidates=n_sobol_starts,
            random_state=random_state + 10_000,
        )
        if n_sobol_starts > 0
        else np.empty((0, dim))
    )

    starts = np.vstack([
        observed_starts,
        sobol_starts,
    ])

    nn_candidates = []

    for fitted_nn in ensemble:
        for x_start in starts:
            candidate = optimise_nn_input(
                fitted_nn=fitted_nn,
                x_start=x_start,
                n_steps=n_steps,
                learning_rate=learning_rate,
            )

            nn_candidates.append(
                candidate
            )

    if not nn_candidates:
        return np.empty((0, dim))

    return np.unique(
        np.round(
            np.asarray(nn_candidates),
            10,
        ),
        axis=0,
    )



# 8. ALTERNATIVE MODELS
#--------------------------------------------------

def fit_extra_trees(
    X: np.ndarray,
    y_model: np.ndarray,
    random_state: int,
) -> ExtraTreesRegressor:
    """
    Fit Extra Trees on transformed outputs.
    """
    model = ExtraTreesRegressor(
        n_estimators=750,
        min_samples_leaf=1,
        max_features=1.0,
        bootstrap=False,
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(
        X,
        y_model,
    )

    return model


def fit_xgboost(
    X: np.ndarray,
    y_model: np.ndarray,
    random_state: int,
) -> Optional[Any]:
    """
    Fit conservative XGBoost model on transformed outputs.
    """
    if not XGBOOST_AVAILABLE:
        return None

    model = XGBRegressor(
        n_estimators=500,
        max_depth=2,
        learning_rate=0.02,
        min_child_weight=2,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(
        X,
        y_model,
    )

    return model



# 10. FULL PIPELINE
#--------------------------------------------------

def run_function_pipeline(
    df: pd.DataFrame,
    config: FunctionConfig,
    random_state: int = 42,
    top_n: int = 20,
) -> Dict[str, Any]:
    """
    Run the complete optimisation pipeline for one function.
    """
    X, y_original, input_columns = extract_xy(
        df
    )

    dim = X.shape[1]

    y_model, transform_info = (
        fit_output_transform(
            y_original,
            config.transform,
        )
    )

    best_index = int(
        np.argmax(y_original)
    )

    x_best = X[
        best_index
    ]

    y_best_original = float(
        y_original[best_index]
    )

    y_best_model = float(
        y_model[best_index]
    )

    # -------------------------
    # Fit GP
    # -------------------------
    gp = fit_gp(
        X=X,
        y_model=y_model,
        noise_level=config.gp_noise_level,
        n_restarts=config.gp_restarts,
        random_state=random_state,
    )

    # -------------------------
    # Fit NN ensemble
    # -------------------------
    nn_ensemble = None
    nn_candidates = np.empty(
        (0, dim)
    )

    if config.use_nn_generator:
        nn_ensemble = fit_nn_ensemble(
            X=X,
            y_model=y_model,
            seeds=[
                random_state + seed
                for seed in config.nn_seeds
            ],
            hidden_sizes=config.nn_hidden_sizes,
            epochs=config.nn_epochs,
            learning_rate=config.nn_learning_rate,
            weight_decay=config.nn_weight_decay,
        )

        nn_candidates = generate_nn_candidates(
            ensemble=nn_ensemble,
            X=X,
            y_original=y_original,
            n_best_starts=config.nn_best_starts,
            n_sobol_starts=config.nn_sobol_starts,
            n_steps=config.nn_gradient_steps,
            learning_rate=(
                config.nn_gradient_learning_rate
            ),
            random_state=random_state,
        )

    # -------------------------
    # Fit auxiliary models
    # -------------------------
    extra_trees = fit_extra_trees(
        X=X,
        y_model=y_model,
        random_state=random_state,
    )

    xgboost_model = fit_xgboost(
        X=X,
        y_model=y_model,
        random_state=random_state,
    )

    # -------------------------
    # Generate candidate pool
    # -------------------------
    global_candidates = (
        generate_sobol_candidates(
            dim=dim,
            n_candidates=config.n_global,
            random_state=random_state,
        )
    )

    local_candidates = (
        generate_local_candidates(
            x_best=x_best,
            n_candidates=config.n_local,
            scale=config.local_scale,
            random_state=random_state + 1,
        )
    )

    candidate_groups = [
        (
            "sobol_global",
            global_candidates,
        ),
        (
            "local_incumbent",
            local_candidates,
        ),
        (
            "nn_gradient",
            nn_candidates,
        ),
        (
            "incumbent",
            np.atleast_2d(x_best),
        ),
    ]

    if config.special_candidates is not None:
        special = np.atleast_2d(
            np.asarray(
                config.special_candidates,
                dtype=float,
            )
        )

        if special.shape[1] != dim:
            raise ValueError(
                "special_candidates have the wrong dimension."
            )

        candidate_groups.append(
            (
                "special",
                special,
            )
        )

    candidate_df = (
        combine_candidate_sources(
            candidate_groups
        )
    )

    candidate_columns = [
        f"x{i + 1}"
        for i in range(dim)
    ]

    candidates = candidate_df[
        candidate_columns
    ].to_numpy(dtype=float)

    # -------------------------
    # Score with GP
    # -------------------------
    gp_mu, gp_sigma = gp.predict(
        candidates,
        return_std=True,
    )

    if config.acquisition == "ei":
        gp_acquisition = expected_improvement(
            mu=gp_mu,
            sigma=gp_sigma,
            incumbent=y_best_model,
            xi=config.xi,
        )

    elif config.acquisition == "ucb":
        gp_acquisition = upper_confidence_bound(
            mu=gp_mu,
            sigma=gp_sigma,
            kappa=config.kappa,
        )

    else:
        raise ValueError(
            "acquisition must be 'ei' or 'ucb'."
        )

    # -------------------------
    # Score with auxiliary models
    # -------------------------
    et_pred = extra_trees.predict(
        candidates
    )

    if xgboost_model is not None:
        xgb_pred = xgboost_model.predict(
            candidates
        )
    else:
        xgb_pred = np.full(
            len(candidates),
            np.nan,
        )

    if nn_ensemble is not None:
        nn_pred, nn_seed_std = (
            predict_nn_ensemble(
                ensemble=nn_ensemble,
                X=candidates,
            )
        )
    else:
        nn_pred = np.full(
            len(candidates),
            np.nan,
        )

        nn_seed_std = np.full(
            len(candidates),
            np.nan,
        )

    final_score = gp_acquisition

    # -------------------------
    # Build candidate report
    # -------------------------
    candidate_df["gp_mu_model"] = gp_mu
    candidate_df["gp_sigma_model"] = gp_sigma
    candidate_df["gp_acquisition"] = gp_acquisition
    candidate_df["nn_pred_model"] = nn_pred
    candidate_df["nn_seed_std"] = nn_seed_std
    candidate_df["extra_trees_pred_model"] = et_pred
    candidate_df["xgboost_pred_model"] = xgb_pred
    candidate_df["final_score"] = final_score

    # Approximate original-unit predictions.
    candidate_df["gp_mu_original"] = (
        inverse_output_transform(
            gp_mu,
            transform_info,
        )
    )

    candidate_df["nn_pred_original"] = (
        inverse_output_transform(
            nn_pred,
            transform_info,
        )
        if nn_ensemble is not None
        else np.nan
    )

    candidate_df[
        "extra_trees_pred_original"
    ] = inverse_output_transform(
        et_pred,
        transform_info,
    )

    candidate_df[
        "xgboost_pred_original"
    ] = (
        inverse_output_transform(
            xgb_pred,
            transform_info,
        )
        if xgboost_model is not None
        else np.nan
    )

    candidate_df[
        "gp_improvement_original_approx"
    ] = (
        candidate_df["gp_mu_original"]
        - y_best_original
    )

    # Avoid resubmitting the incumbent
    incumbent_mask = np.all(
        np.isclose(
            candidates,
            x_best,
            atol=1e-10,
        ),
        axis=1,
    )

    candidate_df[
        "eligible_for_submission"
    ] = ~incumbent_mask

    eligible = candidate_df[
        candidate_df[
            "eligible_for_submission"
        ]
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "No eligible candidates remain after removing the incumbent."
        )

    eligible = eligible.sort_values(
        "final_score",
        ascending=False,
    )

    recommendation = eligible.iloc[0]

    top_candidates = (
        eligible
        .head(top_n)
        .reset_index(drop=True)
    )

    recommended_x = recommendation[
        candidate_columns
    ].to_numpy(dtype=float)

    result = {
        "recommended_x": recommended_x.tolist(),
        "candidate_source": recommendation["source"],
        "acquisition": config.acquisition,
        "final_score": float(
            recommendation["final_score"]
        ),
        "gp_acquisition": float(
            recommendation["gp_acquisition"]
        ),
        "gp_mu_model": float(
            recommendation["gp_mu_model"]
        ),
        "gp_sigma_model": float(
            recommendation["gp_sigma_model"]
        ),
        "gp_mu_original_approx": float(
            recommendation["gp_mu_original"]
        ),
        "nn_pred_original_approx": (
            float(
                recommendation[
                    "nn_pred_original"
                ]
            )
            if nn_ensemble is not None
            else np.nan
        ),
        "extra_trees_pred_original_approx": float(
            recommendation[
                "extra_trees_pred_original"
            ]
        ),
        "xgboost_pred_original_approx": (
            float(
                recommendation[
                    "xgboost_pred_original"
                ]
            )
            if xgboost_model is not None
            else np.nan
        ),
        "best_observed_x": x_best.tolist(),
        "best_observed_y": y_best_original,
        "transform": transform_info["name"],
        "transform_info": transform_info,
        "kernel_summary": str(gp.kernel_),
        "lengthscales": (
            extract_gp_lengthscales(gp)
            .tolist()
        ),
        "n_total_candidates": len(
            candidate_df
        ),
        "n_nn_candidates": len(
            nn_candidates
        ),
        "top_candidates": top_candidates,
        "all_candidates": candidate_df,
        "models": {
            "gp": gp,
            "nn_ensemble": nn_ensemble,
            "extra_trees": extra_trees,
            "xgboost": xgboost_model,
        },
    }

    return result


# 10. RUN All FUNCTIONS
#--------------------------------------------------

def run_all_functions(
    function_data: Dict[str, pd.DataFrame],
    function_configs: Dict[
        str,
        FunctionConfig,
    ],
    base_random_state: int = 42,
    top_n: int = 20,
) -> Tuple[
    Dict[str, Dict[str, Any]],
    pd.DataFrame,
]:
    """
    Run the hybrid pipeline for all functions.
    """
    results = {}
    summary_rows = []

    for function_number, function_name in enumerate(
        function_configs,
        start=1,
    ):
        if function_name not in function_data:
            print(
                f"{function_name} not found; skipping."
            )
            continue

        print(
            f"Running {function_name}..."
        )

        result = run_function_pipeline(
            df=function_data[function_name],
            config=function_configs[
                function_name
            ],
            random_state=(
                base_random_state
                + function_number
            ),
            top_n=top_n,
        )

        results[
            function_name
        ] = result

        summary_rows.append({
            "function": function_name,
            "acquisition": result["acquisition"],
            "transform": result["transform"],
            "source": result["candidate_source"],
            "best_observed_y":
                result["best_observed_y"],
            "gp_predicted_y":
                result[
                    "gp_mu_original_approx"
                ],
            "nn_predicted_y":
                result[
                    "nn_pred_original_approx"
                ],
            "extra_trees_predicted_y":
                result[
                    "extra_trees_pred_original_approx"
                ],
            "xgboost_predicted_y":
                result[
                    "xgboost_pred_original_approx"
                ],
            "gp_sigma_model":
                result["gp_sigma_model"],
            "recommended_x":
                result["recommended_x"],
            "lengthscales":
                result["lengthscales"],
            "n_candidates":
                result["n_total_candidates"],
            "n_nn_candidates":
                result["n_nn_candidates"],
        })

    summary = pd.DataFrame(
        summary_rows
    )

    return results, summary

# Step 4: Uncertainty diagnostics to inform hyperparameters

In [ ]:
uncertainty_rows = []

for n in range(1, 9):

    fname = f"function_{n}"

    result = results[fname]
    candidates = result["all_candidates"]

    y_original = function_data[
        fname
    ]["output"].to_numpy(dtype=float)

    config = FUNCTION_CONFIGS[fname]

    # Apply the same transformation used to fit the GP
    y_model, transform_info = fit_output_transform(
        y=y_original,
        transform=config.transform,
    )

    y_model_sd = max(
        float(np.std(y_model)),
        1e-8,
    )

    recommended_sigma = result[
        "gp_sigma_model"
    ]

    uncertainty_rows.append({
        "function": fname,
        "transform": config.transform,

        "observed_y_sd_original":
            np.std(y_original),

        "observed_y_sd_model":
            y_model_sd,

        "candidate_sigma_median":
            candidates[
                "gp_sigma_model"
            ].median(),

        "candidate_sigma_90th":
            candidates[
                "gp_sigma_model"
            ].quantile(0.90),

        "candidate_sigma_max":
            candidates[
                "gp_sigma_model"
            ].max(),

        "recommended_sigma":
            recommended_sigma,

        "recommended_sigma_scaled":
            recommended_sigma / y_model_sd,
    })


uncertainty_summary = (
    pd.DataFrame(uncertainty_rows)
    .set_index("function")
)

SyntaxError: incomplete input (640996629.py, line 66)

In [ ]:
# calculate percentiles

percentile_rows = []

for n in range(1, 9):

    fname = f"function_{n}"

    candidates = results[
        fname
    ]["all_candidates"]

    recommended_sigma = results[
        fname
    ]["gp_sigma_model"]

    sigma_percentile = (
        candidates["gp_sigma_model"]
        .le(recommended_sigma)
        .mean()
        * 100
    )

    percentile_rows.append({
        "function": fname,
        "recommended_sigma": recommended_sigma,
        "sigma_percentile": sigma_percentile,
    })


sigma_percentiles = (
    pd.DataFrame(percentile_rows)
    .set_index("function")
)

display(
    sigma_percentiles.round(2)
)

# recommend changes to hyperparameter kappa
# uncertainty may be over-weighted in later stages

def recommend_acquisition_hyperparameter(
    sigma_percentile: float,
    sigma_scaled: float | None = None,
    acquisition: str = "ucb",
) -> float:
    """
    Recommend kappa for UCB or xi for EI using:

    1. The uncertainty percentile of the recommended candidate.
    2. The candidate's GP uncertainty relative to observed output
       variation in model space.

    Parameters
    ----------
    sigma_percentile:
        Percentile rank of uncertainty at the recommended candidate,
        measured from 0 to 100.

    sigma_scaled:
        Recommended GP sigma divided by the standard deviation of the
        observed outputs in model space.

    acquisition:
        Either "ucb" or "ei".

    Returns
    -------
    float
        Recommended kappa or xi.
    """
    acquisition = acquisition.lower()

    if not 0 <= sigma_percentile <= 100:
        raise ValueError(
            "sigma_percentile must lie between 0 and 100."
        )

    if sigma_scaled is not None and sigma_scaled < 0:
        raise ValueError(
            "sigma_scaled must be non-negative."
        )

    # --------------------------------------------------------
    # Initial exploration classification
    # --------------------------------------------------------

    if sigma_percentile >= 80:
        exploration_level = "high"

    elif sigma_percentile >= 60:
        exploration_level = "moderate_high"

    elif sigma_percentile >= 30:
        exploration_level = "moderate"

    elif sigma_percentile >= 10:
        exploration_level = "low"

    else:
        exploration_level = "very_low"

    # --------------------------------------------------------
    # Adjust for absolute uncertainty
    # --------------------------------------------------------

    if sigma_scaled is not None:

        # Candidate is relatively uncertain, but absolute
        # uncertainty is actually small.
        if sigma_scaled < 0.10:

            exploration_level = {
                "high": "moderate_high",
                "moderate_high": "moderate",
                "moderate": "low",
                "low": "very_low",
                "very_low": "very_low",
            }[exploration_level]

        # Absolute uncertainty remains genuinely high.
        elif sigma_scaled > 0.50:

            exploration_level = {
                "very_low": "low",
                "low": "moderate",
                "moderate": "moderate_high",
                "moderate_high": "high",
                "high": "high",
            }[exploration_level]

    # --------------------------------------------------------
    # Return acquisition-specific parameter
    # --------------------------------------------------------

    if acquisition == "ucb":

        kappa_map = {       # during later stages where uncertainty is very high, lower kappa to avoid random exploration
            "very_low": 1.50,
            "low": 1.25,
            "moderate": 1.00,
            "moderate_high": 0.75,
            "high": 0.50,
            }

        return kappa_map[exploration_level]

    if acquisition == "ei":

        xi_map = {
            "very_low": 0.000,
            "low": 0.005,
            "moderate": 0.010,
            "moderate_high": 0.025,
            "high": 0.050,
        }

        return xi_map[exploration_level]

    raise ValueError(
        "acquisition must be either 'ucb' or 'ei'."
    )


# Combine with scaled diagnostics

hyperparameter_diagnostics = (
    sigma_percentiles
    .join(
        uncertainty_summary[
            ["recommended_sigma_scaled"]
        ],
        how="left",
    )
)

if hyperparameter_diagnostics[
    "recommended_sigma_scaled"
].isna().any():

    missing_functions = (
        hyperparameter_diagnostics[
            hyperparameter_diagnostics[
                "recommended_sigma_scaled"
            ].isna()
        ]
        .index
        .tolist()
    )

    raise ValueError(
        "Missing scaled-sigma diagnostics for: "
        f"{missing_functions}"
    )


# Suggest new kappa

hyperparameter_diagnostics[
    "recommended_kappa"
] = hyperparameter_diagnostics.apply(
    lambda row: recommend_acquisition_hyperparameter(
        sigma_percentile=row["sigma_percentile"],
        sigma_scaled=row["recommended_sigma_scaled"],
        acquisition="ucb",
    ),
    axis=1,
)

hyperparameter_diagnostics[
    "recommended_xi"
] = hyperparameter_diagnostics.apply(
    lambda row: recommend_acquisition_hyperparameter(
        sigma_percentile=row["sigma_percentile"],
        sigma_scaled=row["recommended_sigma_scaled"],
        acquisition="ei",
    ),
    axis=1,
)


display(
    hyperparameter_diagnostics[
        [
            "recommended_sigma",
            "sigma_percentile",
            "recommended_sigma_scaled",
            "recommended_kappa",
            "recommended_xi",
        ]
    ].round(4)
)

,recommended_sigma,sigma_percentile,recommended_sigma_scaled,recommended_kappa,recommended_xi
function,,,,,
function_1,1.8811,0.6530,0.6644,1.00,0.005
function_2,0.0673,29.4110,0.2544,1.00,0.005
function_3,0.0201,30.3937,0.1933,1.25,0.010
function_4,0.3759,6.1357,0.0380,0.75,0.000
function_5,0.4111,66.5111,0.2007,1.50,0.025
function_6,0.0733,64.5713,0.1283,1.50,0.025
function_7,0.1419,0.5920,0.2499,0.75,0.000
function_8,0.0927,74.5864,0.0791,1.25,0.010


#Step 5: Get next Recommendation

In [ ]:

results, summary = run_all_functions(
    function_data=function_data,
    function_configs=FUNCTION_CONFIGS,
    base_random_state=42,
    top_n=20,
)



display_columns = [
    "function",
    "acquisition",
    "xi",
    "kappa",
    "transform",
    "source",
    "best_observed_y",
    "gp_predicted_y",
    "gp_sigma_model",
    "recommended_x",
    "lengthscales",
    "n_candidates",
    "n_previously_evaluated_removed",
]

available_columns = [
    column
    for column in display_columns
    if column in summary.columns
]

summary_table = (
    summary[available_columns]
    .copy()
    .set_index("function")
    .round(4)
)

display(summary_table)

# Format correctly into Capstone portal
#---------------------------------------------------

formatted_points = []

for recommended_x in summary["recommended_x"]:

    formatted_coords = [
        f"{coord:.6f}"
        for coord in recommended_x
    ]

    formatted_points.append(
        "-".join(formatted_coords)
    )


# Display table
display_df = pd.DataFrame({
    "Function": summary["function"],
    "Suggested Next Point": formatted_points,
}).set_index("Function")


print("Next Suggested Points:")

display(display_df)

Next Suggested Points:


,Suggested Next Point
Function,
function_1,0.368064-0.364038
function_2,0.702546-0.981348
function_3,0.978200-0.924638-0.488672
function_4,0.375935-0.416533-0.369314-0.411327
function_5,0.790110-1.000000-1.000000-1.000000
function_6,0.500919-0.270238-0.522920-0.656931-0.000805
function_7,0.018185-0.112915-0.524584-0.133992-0.361762-0...
function_8,0.002002-0.110261-0.073770-0.258387-0.998143-0...


# Step 6: Diagnostics to check GP recomendations vs Trees/NN

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Optional XGBoost import
# ------------------------------------------------------------

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    warnings.warn(
        "xgboost is not installed. "
        "The comparison will run without XGBoost."
    )


# Configuration
# ------------------------------------------------------------

TOP_N = 10
RANDOM_STATE = 42

NN_SEEDS = [11, 22, 33, 44, 55]


# Fit a neural-network ensemble
# ------------------------------------------------------------

def fit_nn_ensemble(
    X: np.ndarray,
    y: np.ndarray,
    seeds=NN_SEEDS,
):
    """
    Fit several small MLP regressors using different random seeds.

    Each model standardises the inputs internally. MLPRegressor also
    handles output scaling reasonably for many datasets, although for
    very skewed outputs a separate output transformation may help.
    """
    models = []

    for seed in seeds:
        model = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(16, 16),
                activation="tanh",
                solver="lbfgs",
                alpha=0.01,
                max_iter=5000,
                random_state=seed,
            ),
        )

        model.fit(X, y)
        models.append(model)

    return models


def predict_nn_ensemble(
    models,
    X_new: np.ndarray,
):
    """
    Return the ensemble mean and seed-to-seed standard deviation.
    """
    predictions = np.column_stack([
        model.predict(X_new)
        for model in models
    ])

    return (
        predictions.mean(axis=1),
        predictions.std(axis=1),
    )

# Obtain GP candidates from the revised pipeline
# ------------------------------------------------------------

def get_gp_top_candidates(
    result: dict,
    dim: int,
    top_n: int = 10,
) -> pd.DataFrame:
    """
    Extract the top candidate table from the revised BO pipeline.
    """
    if "top_candidates" not in result:
        raise KeyError(
            "The result does not contain 'top_candidates'. "
            "Check that you are using the revised pipeline."
        )

    top_candidates = result["top_candidates"].copy()

    candidate_cols = [
        f"x{i + 1}"
        for i in range(dim)
    ]

    missing_columns = [
        column
        for column in candidate_cols
        if column not in top_candidates.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Candidate coordinate columns are missing: "
            f"{missing_columns}"
        )

    top_candidates = (
        top_candidates
        .head(top_n)
        .reset_index(drop=True)
    )

    top_candidates.insert(
        0,
        "rank",
        np.arange(1, len(top_candidates) + 1),
    )

    return top_candidates



# Run diagnostics for all functions
# ------------------------------------------------------------

comparison_results = {}

for n in range(1, 9):

    fname = f"function_{n}"

    print("\n" + "=" * 100)
    print(fname.upper())
    print("=" * 100)

    try:

        # Historical observations

        df = function_data[fname].copy()

        if "output" not in df.columns:
            raise ValueError(
                f"{fname} does not contain an 'output' column."
            )

        input_columns = [
            column
            for column in df.columns
            if column != "output"
        ]

        X = df[input_columns].to_numpy(dtype=float)
        y = df["output"].to_numpy(dtype=float)

        dim = X.shape[1]

        candidate_cols = [
            f"x{i + 1}"
            for i in range(dim)
        ]


        # GP top candidates

        top_candidates = get_gp_top_candidates(
            result=results[fname],
            dim=dim,
            top_n=TOP_N,
        )

        X_candidates = top_candidates[
            candidate_cols
        ].to_numpy(dtype=float)


        # Extra Trees

        extra_trees = ExtraTreesRegressor(
            n_estimators=500,
            max_features="sqrt",
            min_samples_leaf=2,
            bootstrap=False,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        extra_trees.fit(X, y)

        top_candidates["extra_trees_pred"] = (
            extra_trees.predict(X_candidates)
        )


        # XGBoost

        if XGBOOST_AVAILABLE:
            xgboost_model = XGBRegressor(
                n_estimators=150,
                max_depth=2,
                learning_rate=0.03,
                min_child_weight=2,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.05,
                reg_lambda=5.0,
                objective="reg:squarederror",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )

            xgboost_model.fit(X, y)

            top_candidates["xgboost_pred"] = (
                xgboost_model.predict(X_candidates)
            )

        else:
            top_candidates["xgboost_pred"] = np.nan


        # Neural-network ensemble

        nn_models = fit_nn_ensemble(
            X=X,
            y=y,
        )

        nn_mean, nn_std = predict_nn_ensemble(
            models=nn_models,
            X_new=X_candidates,
        )

        top_candidates["nn_pred"] = nn_mean
        top_candidates["nn_seed_std"] = nn_std

        # Identify GP prediction column

        if "gp_mu_original" in top_candidates.columns:
            gp_prediction_column = "gp_mu_original"

        elif "gp_mu_original_approx" in top_candidates.columns:
            gp_prediction_column = "gp_mu_original_approx"

        else:
            raise KeyError(
                "No original-scale GP prediction column found. "
                "Expected 'gp_mu_original' or "
                "'gp_mu_original_approx'."
            )


        # Pairwise differences

        top_candidates["gp_minus_extra_trees"] = (
            top_candidates[gp_prediction_column]
            - top_candidates["extra_trees_pred"]
        )

        top_candidates["gp_minus_xgboost"] = (
            top_candidates[gp_prediction_column]
            - top_candidates["xgboost_pred"]
        )

        top_candidates["gp_minus_nn"] = (
            top_candidates[gp_prediction_column]
            - top_candidates["nn_pred"]
        )


        # Cross-model average and disagreement

        model_prediction_columns = [
            gp_prediction_column,
            "extra_trees_pred",
            "nn_pred",
        ]

        if XGBOOST_AVAILABLE:
            model_prediction_columns.append(
                "xgboost_pred"
            )

        prediction_matrix = top_candidates[
            model_prediction_columns
        ].to_numpy(dtype=float)

        top_candidates["model_mean"] = (
            np.nanmean(
                prediction_matrix,
                axis=1,
            )
        )

        top_candidates["model_std"] = (
            np.nanstd(
                prediction_matrix,
                axis=1,
            )
        )

        top_candidates["model_range"] = (
            np.nanmax(
                prediction_matrix,
                axis=1,
            )
            - np.nanmin(
                prediction_matrix,
                axis=1,
            )
        )

        # Scale disagreement relative to variation in observed y.
        y_scale = max(
            float(np.std(y)),
            1e-8,
        )

        top_candidates["model_std_scaled"] = (
            top_candidates["model_std"]
            / y_scale
        )


        # Model-specific rankings

        top_candidates["gp_prediction_rank"] = (
            top_candidates[
                gp_prediction_column
            ]
            .rank(
                ascending=False,
                method="min",
            )
            .astype(int)
        )

        top_candidates["extra_trees_rank"] = (
            top_candidates[
                "extra_trees_pred"
            ]
            .rank(
                ascending=False,
                method="min",
            )
            .astype(int)
        )

        top_candidates["nn_rank"] = (
            top_candidates[
                "nn_pred"
            ]
            .rank(
                ascending=False,
                method="min",
            )
            .astype(int)
        )

        if XGBOOST_AVAILABLE:
            top_candidates["xgboost_rank"] = (
                top_candidates[
                    "xgboost_pred"
                ]
                .rank(
                    ascending=False,
                    method="min",
                )
                .astype(int)
            )
        else:
            top_candidates["xgboost_rank"] = np.nan

        # Average rank is a diagnostic only.
        ranking_columns = [
            "gp_prediction_rank",
            "extra_trees_rank",
            "nn_rank",
        ]

        if XGBOOST_AVAILABLE:
            ranking_columns.append(
                "xgboost_rank"
            )

        top_candidates["average_model_rank"] = (
            top_candidates[
                ranking_columns
            ].mean(axis=1)
        )


        # Store complete result

        comparison_results[fname] = {
            "comparison_table": top_candidates,
            "models": {
                "extra_trees": extra_trees,
                "xgboost": (
                    xgboost_model
                    if XGBOOST_AVAILABLE
                    else None
                ),
                "neural_networks": nn_models,
            },
        }


        # Display compact table

        candidate_cols = [
            f"x{i + 1}"
            for i in range(dim)
        ]


        columns_to_display = [
            "rank",
            gp_prediction_column,
            "gp_sigma_model",
            "gp_acquisition",
            "extra_trees_pred",
            "xgboost_pred",
            "nn_pred",
            "nn_seed_std",
            "model_mean",
            "model_std",
            "model_std_scaled",
            "average_model_rank",
            "source",
        ]

        columns_to_display = [
            column
            for column in columns_to_display
            if column in top_candidates.columns
        ]

        display(
            top_candidates[
                columns_to_display
            ].round(4)
        )

    except Exception as e:
        # Print the error for debugging, but allow the loop to continue
        print(f"An error occurred while processing {fname}: {e}")
        import traceback
        traceback.print_exc()


FUNCTION_1


,rank,gp_mu_original,gp_sigma_model,gp_acquisition,extra_trees_pred,xgboost_pred,nn_pred,nn_seed_std,model_mean,model_std,model_std_scaled,average_model_rank,source
0,1,0.0,1.8811,4.9458,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1418,4.25,local_incumbent
1,2,0.0,1.9844,4.9311,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1413,3.50,local_incumbent
2,3,0.0,1.9986,4.9180,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1413,5.50,local_incumbent
3,4,0.0,1.5151,4.8961,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1431,4.00,local_incumbent
4,5,0.0,1.5786,4.8933,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1428,2.25,local_incumbent
5,6,0.0,1.5049,4.8931,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1431,3.25,local_incumbent
6,7,0.0,2.0721,4.8906,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1410,6.25,local_incumbent
7,8,0.0,2.0960,4.8891,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1408,5.25,local_incumbent
8,9,0.0,1.8238,4.8876,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1416,2.50,local_incumbent
9,10,0.0,2.0515,4.8865,-0.0001,-0.0002,-0.0003,0.0001,-0.0002,0.0001,0.1410,6.50,local_incumbent



FUNCTION_2


,rank,gp_mu_original,gp_sigma_model,gp_acquisition,extra_trees_pred,xgboost_pred,nn_pred,nn_seed_std,model_mean,model_std,model_std_scaled,average_model_rank,source
0,1,0.6038,0.0684,0.7407,0.3131,0.4520,0.6043,0.0577,0.4933,0.1212,0.4579,3.50,sobol_global
1,2,0.5990,0.0707,0.7404,0.3125,0.4476,0.6186,0.0593,0.4944,0.1242,0.4692,6.00,sobol_global
2,3,0.5994,0.0705,0.7404,0.3125,0.4476,0.6133,0.0521,0.4932,0.1229,0.4646,6.00,sobol_global
3,4,0.6033,0.0685,0.7403,0.3125,0.4476,0.5851,0.0247,0.4871,0.1174,0.4437,4.50,sobol_global
4,5,0.6016,0.0692,0.7400,0.3100,0.4476,0.5712,0.0234,0.4826,0.1151,0.4351,8.00,sobol_global
5,6,0.6027,0.0685,0.7398,0.3065,0.4476,0.5531,0.0434,0.4775,0.1135,0.4290,8.25,sobol_global
6,7,0.6033,0.0682,0.7398,0.4499,0.5054,0.5840,0.0073,0.5356,0.0616,0.2328,2.75,local_incumbent
7,8,0.6029,0.0684,0.7398,0.4499,0.5054,0.5835,0.0070,0.5354,0.0614,0.2321,3.75,local_incumbent
8,9,0.6029,0.0684,0.7398,0.4499,0.5054,0.5834,0.0068,0.5354,0.0614,0.2320,3.75,local_incumbent
9,10,0.6033,0.0682,0.7398,0.4499,0.5054,0.5836,0.0066,0.5356,0.0615,0.2326,2.75,local_incumbent



FUNCTION_3


...

,rank,gp_mu_original,gp_sigma_model,gp_acquisition,extra_trees_pred,xgboost_pred,nn_pred,nn_seed_std,model_mean,model_std,model_std_scaled,average_model_rank,source
0,1,-0.3388,0.0673,-0.2716,-0.7591,-0.8435,-0.2383,0.0165,-0.5449,0.2606,0.4561,6.00,local_incumbent
1,2,-0.3329,0.0612,-0.2717,-0.7313,-0.8435,-0.2373,0.0146,-0.5363,0.2565,0.4490,3.50,local_incumbent
2,3,-0.3297,0.0578,-0.2719,-0.7346,-0.8435,-0.2456,0.0135,-0.5384,0.2554,0.4470,5.00,local_incumbent
3,4,-0.3317,0.0598,-0.2719,-0.7528,-0.8435,-0.2449,0.0144,-0.5432,0.2588,0.4530,6.25,local_incumbent
4,5,-0.3452,0.0733,-0.2719,-0.7792,-0.8435,-0.2283,0.0164,-0.5491,0.2665,0.4665,6.25,local_incumbent
5,6,-0.3290,0.0570,-0.2720,-0.7436,-0.8328,-0.2400,0.0128,-0.5364,0.2558,0.4477,3.25,local_incumbent
6,7,-0.3275,0.0553,-0.2722,-0.7460,-0.8328,-0.2404,0.0130,-0.5367,0.2565,0.4489,3.50,local_incumbent
7,8,-0.3428,0.0706,-0.2722,-0.7745,-0.8435,-0.2283,0.0155,-0.5473,0.2660,0.4655,6.00,local_incumbent
8,9,-0.3295,0.0573,-0.2722,-0.7410,-0.8328,-0.2434,0.0145,-0.5367,0.2542,0.4449,3.75,local_incumbent
9,10,-0.3361,0.0639,-0.2723,-0.7433,-0.8435,-0.2404,0.0162,-0.5408,0.2573,0.4503,5.50,local_incumbent



FUNCTION_7


,rank,gp_mu_original,gp_sigma_model,gp_acquisition,extra_trees_pred,xgboost_pred,nn_pred,nn_seed_std,model_mean,model_std,model_std_scaled,average_model_rank,source
0,1,2.1255,0.1419,2.2674,1.3375,1.4131,2.4411,0.0261,1.8293,0.4683,0.8248,4.75,local_incumbent
1,2,2.1257,0.1411,2.2668,1.3793,1.3665,2.1182,0.0151,1.7474,0.3746,0.6598,5.25,local_incumbent
2,3,2.1245,0.1420,2.2666,1.3932,1.4131,2.3925,0.0109,1.8308,0.4381,0.7717,5.25,local_incumbent
3,4,2.1254,0.1407,2.2662,1.4182,1.3619,2.0955,0.0310,1.7503,0.3609,0.6357,5.50,local_incumbent
4,5,2.1246,0.1415,2.2661,1.3101,1.3740,2.4629,0.0061,1.8179,0.4912,0.8652,6.00,local_incumbent
5,6,2.1243,0.1417,2.2660,1.3705,1.3463,2.5278,0.0168,1.8422,0.5045,0.8887,6.50,local_incumbent
6,7,2.1249,0.1409,2.2658,1.3612,1.3464,1.9798,0.0493,1.7031,0.3531,0.6219,7.75,local_incumbent
7,8,2.1225,0.1433,2.2658,1.4110,1.4206,2.3377,0.0088,1.8229,0.4142,0.7296,5.50,local_incumbent
8,9,2.1245,0.1401,2.2647,1.3643,1.4483,2.4099,0.0148,1.8368,0.4431,0.7805,5.00,local_incumbent
9,10,2.1266,0.1380,2.2645,1.2613,1.4512,2.5328,0.0620,1.8430,0.5119,0.9017,3.25,local_incumbent



FUNCTION_8


,rank,gp_mu_original,gp_sigma_model,gp_acquisition,extra_trees_pred,xgboost_pred,nn_pred,nn_seed_std,model_mean,model_std,model_std_scaled,average_model_rank,source
0,1,9.9137,0.0927,10.0063,9.4088,9.4103,9.9224,0.0461,9.6638,0.2543,0.2171,5.25,local_incumbent
1,2,9.9222,0.0794,10.0016,9.3645,9.2655,9.9529,0.0238,9.6262,0.3134,0.2676,5.75,local_incumbent
2,3,9.9173,0.0833,10.0006,9.3848,9.4103,9.8986,0.0334,9.6528,0.2554,0.2181,5.75,local_incumbent
3,4,9.9320,0.0685,10.0004,9.4090,9.2938,9.9574,0.0197,9.6480,0.2995,0.2558,3.25,local_incumbent
4,5,9.8785,0.1210,9.9994,9.0276,9.2294,9.8623,0.0311,9.4995,0.3778,0.3226,10.00,sobol_global
5,6,9.9238,0.0747,9.9984,9.3440,9.2535,9.9387,0.0188,9.6150,0.3179,0.2714,6.50,local_incumbent
6,7,9.9174,0.0802,9.9977,9.4387,9.3058,9.9279,0.0339,9.6475,0.2792,0.2384,4.75,local_incumbent
7,8,9.8931,0.1045,9.9976,9.4008,9.4103,9.9529,0.0215,9.6643,0.2596,0.2216,4.50,local_incumbent
8,9,9.9026,0.0944,9.9971,9.4305,9.4103,9.9326,0.0270,9.6690,0.2489,0.2125,4.25,local_incumbent
9,10,9.9280,0.0687,9.9966,9.4229,9.4103,9.9441,0.0194,9.6763,0.2598,0.2218,2.50,local_incumbent
